In [3]:
import requests
import pandas as pd
import unicodedata

df = pd.read_csv("dataa.csv", delimiter = '|')

df.columns

Index(['nom', 'siren', 'adresse'], dtype='object')

In [4]:
df

,nom,siren,adresse
0,MONSIEUR VINCENT CACHERA,518 418 496,"187 RUE DU 11 NOVEMBRE 1918, 38540 VALENCIN"
1,483 072 658,483 072 658,NC
2,TRANSPORTS TESTUD SA,970 500 195,"RTE DE GIVORS ZI LA SAULAIE LD GARENNE, 38670 ..."
3,TRANS LUZAIS AFFRETEMENT,402 488 878,"10 RUE DE LUZAIS, 38070 SAINT-QUENTIN-FALLAVIER"
4,VIENNATRANS SARL,733 680 243,"LEVEAU, 38200 VIENNE"
...,...,...,...
5611,MONSIEUR PAUL MICOD,553 612 334,38490 FITILIEU
5612,MONSIEUR PAUL LEVET,573 675 303,"RUE VICTOR FAUGIER, 38200 VIENNE"
5613,MONSIEUR PAUL GUILLE,593 674 898,"RUE ALBERT THOMAS, 38200 VIENNE"
5614,MONSIEUR PAUL GIROUD,062 405 998,38530 BARRAUX


In [5]:
df_split = df['adresse'].str.split(',').str[1].str.strip().str.split(' ', n=1, expand=True).rename(columns={0: 'code_postal', 1: 'commune'})
df_split = df_split.dropna().drop_duplicates()
df_split['latitude'] = None
df_split['longitude'] = None
df_split

,code_postal,commune,latitude,longitude
0,38540,VALENCIN,None,None
2,38670,CHASSE-SUR-RHONE,None,None
3,38070,SAINT-QUENTIN-FALLAVIER,None,None
4,38200,VIENNE,None,None
5,38130,ECHIROLLES,None,None
...,...,...,...,...
5076,38460,OPTEVOZ,None,None
5085,38730,VALENCOGNE,None,None
5294,38510,LE BOUCHAGE,None,None
5378,38620,VELANNE,None,None


In [7]:


def normaliser(texte):
    texte = texte.lower().replace('-',' ')
    texte = unicodedata.normalize('NFKD', texte).encode('ASCII', 'ignore').decode('utf-8')
    return texte

def get_coords(code_postal, nom):
    url = "https://geo.api.gouv.fr/communes"
    params = {
        "codePostal": code_postal,
        "fields": "nom,centre"
    }
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        data = response.json()
        if data:
            for i in data:
                if normaliser(i['nom']) == normaliser(nom):
                    return (i['centre']['coordinates'])
    return ([None,None])
for index, row in df_split.iterrows():
    try:
        code_postal = str(int(row["code_postal"]))
    except (ValueError, TypeError):
        continue  # ou log l'index pour investiguer plus tard
    lon, lat = get_coords(code_postal, row["commune"])
    df_split.at[index, 'latitude'] = lat
    df_split.at[index, 'longitude'] = lon



In [8]:
df_split


,code_postal,commune,latitude,longitude
0,38540,VALENCIN,45.6058,5.0259
2,38670,CHASSE-SUR-RHONE,45.578,4.8129
3,38070,SAINT-QUENTIN-FALLAVIER,45.6392,5.1128
4,38200,VIENNE,45.5221,4.8803
5,38130,ECHIROLLES,45.1441,5.7148
...,...,...,...,...
5076,38460,OPTEVOZ,45.752,5.3406
5085,38730,VALENCOGNE,45.4893,5.5344
5294,38510,LE BOUCHAGE,45.6689,5.5288
5378,38620,VELANNE,45.4888,5.6441


In [9]:
df_split.isnull().sum()

code_postal     0
commune         0
latitude       14
longitude      14
dtype: int64

In [10]:
df_split[df_split.isnull().any(axis=1)]

,code_postal,commune,latitude,longitude
112,38520,VENOSC,None,None
318,38880,AUTRANS,None,None
439,38630,VEYRINS THUELLIN,None,None
1012,AU,LAVILLA,None,None
1183,38340,POMMIERS LA PLACETTE,None,None
1586,APPT,9,None,None
1962,1ERE,PORTE DE DROITE,None,None
2152,LES,7 LAUX,None,None
2628,LOTISSEMENT,LE VERGER DE CALVILLE,None,None
2852,RESIDENCE,TIME,None,None


In [11]:
df_split.dropna()

,code_postal,commune,latitude,longitude
0,38540,VALENCIN,45.6058,5.0259
2,38670,CHASSE-SUR-RHONE,45.578,4.8129
3,38070,SAINT-QUENTIN-FALLAVIER,45.6392,5.1128
4,38200,VIENNE,45.5221,4.8803
5,38130,ECHIROLLES,45.1441,5.7148
...,...,...,...,...
5076,38460,OPTEVOZ,45.752,5.3406
5085,38730,VALENCOGNE,45.4893,5.5344
5294,38510,LE BOUCHAGE,45.6689,5.5288
5378,38620,VELANNE,45.4888,5.6441


In [12]:
df_split.to_csv("communes.csv", index=False)